# 03 · Validación del dataset de evaluación

**Entrada:** `data/processed/manifests/evaluation_v1.csv`

**Salidas:** `outputs/dataset/dataset_validation.csv` y `outputs/dataset/dataset_validation_summary.txt`

Verifica que el dataset dorado pueda ser consumido por ASR, traducción (NMT) y voice cloning: audios existentes y abribles, sin vacíos, sin duplicados, duraciones válidas y sin *leakage* de hablantes.

## 1. Entorno y dependencias

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT_OVERRIDE: Path | None = None

def _is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _is_project_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'requirements' / 'dataset.txt').is_file()

def _find_drive_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        return PROJECT_ROOT_OVERRIDE.expanduser().resolve()
    matches = sorted({src_dir.parent for src_dir in Path('/content/drive').glob('**/src')
                      if _is_project_root(src_dir.parent)})
    if len(matches) == 1:
        return matches[0]
    if matches:
        found = '\n - '.join(str(path) for path in matches)
        raise SystemExit(f'Se encontraron varios proyectos:\n - {found}\nAsigna uno a PROJECT_ROOT_OVERRIDE.')
    raise SystemExit('No se encontro IA-Proyecto. Monta la cuenta correcta de Drive o asigna PROJECT_ROOT_OVERRIDE.')

if _is_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = _find_drive_project_root()
else:
    PROJECT_ROOT = Path.cwd()

if not _is_project_root(PROJECT_ROOT):
    raise SystemExit(f'No se encontro la raiz del proyecto en {PROJECT_ROOT}.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
%pip install --quiet -r requirements/dataset.txt

### 1.1 Imports

In [ ]:
import pandas as pd
from pathlib import Path
from IPython.display import display

from src.dataset import validation
from src.dataset.manifest import load_manifest, EVALUATION_CSV

## 2. Cargar el dataset

In [ ]:
df = load_manifest(EVALUATION_CSV)
print(f"Filas: {len(df)}")
display(df)

## 3. Chequeo por fila

Para cada audio se verifica: existencia del archivo, apertura (cabecera), sample rate, duración, vacío, presencia de texto y duplicados.

In [ ]:
check = validation.validate_manifest(df, audio_root=PROJECT_ROOT)
display(check)

## 4. Resumen agregado

In [ ]:
leakage = validation.check_speaker_leakage(df)
summary = validation.build_summary(df, check, leakage)
display(summary)

## 5. Distribución de duración

In [ ]:
durations = check["duration_sec"].dropna()
display(durations.describe().to_frame("duración (seg)"))

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots()
    ax.hist(durations, bins=10, edgecolor="white")
    ax.set_title("Distribución de duración de la muestra")
    ax.set_xlabel("segundos")
    ax.set_ylabel("número de audios")
    plt.show()
except ImportError:
    print("matplotlib no disponible; se omite el histograma.")

## 6. Distribución dev/test y hablantes

In [ ]:
display(df["split"].value_counts().rename("count").to_frame())

In [ ]:
if "speaker_id" in df.columns and df["speaker_id"].notna().any():
    display(pd.DataFrame({"hablante": sorted(df["speaker_id"].dropna().unique())}))
else:
    print("No hay speaker_id en este dataset (FLEURS no lo expone).")

## 7. Leakage entre dev/test

El *leakage* de hablante ocurre cuando el mismo hablante aparece en `dev` y `test`, inflando las métricas. En este dataset no hay `speaker_id`, así que no se puede evaluar y se reporta vacío.

In [ ]:
leakage = validation.check_speaker_leakage(df)
display(pd.DataFrame({"hablantes_compartidos_dev_test": leakage or ["ninguno"]}))

## 8. Guardar el reporte de validación

In [ ]:
csv_out, txt_out = validation.save_validation_files(check, summary)
print()
print(Path(txt_out).read_text(encoding="utf-8"))

## 9. Conclusión

Si el resumen no muestra errores, `evaluation_v1.csv` queda listo para las fases siguientes (ASR, traducción y voice cloning). Este reporte de validación es reproducible: al re-ejecutar el notebook se regenera en `outputs/dataset/`.